In [1]:

import sys
import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from scipy.sparse import load_npz
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize


def resolve_code_root() -> Path:
    cwd = Path.cwd().resolve()
    base = cwd.parent if cwd.name == 'notebooks' else cwd
    if (base / 'utils').exists():
        return base
    if (base / 'phase_3_5_starter' / 'utils').exists():
        return base / 'phase_3_5_starter'
    if (Path('/mnt/data/phase_3_5_starter') / 'utils').exists():
        return Path('/mnt/data/phase_3_5_starter')
    return base

CODE_ROOT = resolve_code_root()
DATA_ROOT = CODE_ROOT
if not ((DATA_ROOT / 'data' / 'processed' / 'clean_resume.csv').exists() or (DATA_ROOT / 'data' / 'clean_resume.csv').exists() or (DATA_ROOT / 'clean_resume.csv').exists()):
    DATA_ROOT = CODE_ROOT.parent

sys.path.append(str(CODE_ROOT))

from utils.feature_engineering import (
    FEATURE_COLUMNS,
    assign_multiclass_label,
    build_pair_features,
    hybrid_label_score,
)

DATA_DIR = DATA_ROOT if ((DATA_ROOT / 'clean_resume.csv').exists() or (DATA_ROOT / 'clean_job.csv').exists() or (DATA_ROOT / 'job_descriptions.csv').exists() or (DATA_ROOT / 'evaluation_pairs_multiclass.csv').exists()) else (DATA_ROOT / 'data')
PROCESSED_DIR = DATA_DIR / 'processed' if (DATA_DIR / 'processed').exists() else DATA_DIR
MODEL_DIR = CODE_ROOT / 'models'
SCORE_DIR = DATA_DIR / 'score'
SCORE_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print('CODE_ROOT =', CODE_ROOT)
print('DATA_ROOT =', DATA_ROOT)
print('DATA_DIR =', DATA_DIR)
print('MODEL_DIR =', MODEL_DIR)
print('SCORE_DIR =', SCORE_DIR)


CODE_ROOT = C:\Users\Mr_Root\Downloads\New folder (4)\myproject
DATA_ROOT = C:\Users\Mr_Root\Downloads\New folder (4)\myproject
DATA_DIR = C:\Users\Mr_Root\Downloads\New folder (4)\myproject\data
MODEL_DIR = C:\Users\Mr_Root\Downloads\New folder (4)\myproject\models
SCORE_DIR = C:\Users\Mr_Root\Downloads\New folder (4)\myproject\data\score


In [2]:

def pick_existing(*paths: Path) -> Path:
    for p in paths:
        if p.exists():
            return p
    raise FileNotFoundError(f'Could not find any of: {paths}')

resume_path = pick_existing(PROCESSED_DIR / 'clean_resume.csv', DATA_ROOT / 'clean_resume.csv')
job_path = pick_existing(PROCESSED_DIR / 'clean_job.csv', DATA_ROOT / 'clean_job.csv')
raw_job_path = pick_existing(DATA_DIR / 'job_descriptions.csv', DATA_ROOT / 'job_descriptions.csv')

resume_df = pd.read_csv(resume_path)
job_df = pd.read_csv(job_path)
raw_job_df = pd.read_csv(raw_job_path)

if 'experience' not in job_df.columns and 'Experience' in raw_job_df.columns and len(job_df) == len(raw_job_df):
    job_df['experience'] = raw_job_df['Experience']
if 'clean_experience' not in job_df.columns and 'experience' in job_df.columns:
    job_df['clean_experience'] = job_df['experience'].fillna('').astype(str).str.lower()

required_resume_cols = ['resume_id', 'category', 'clean_resume']
required_job_cols = ['job_id', 'title', 'full_job']
for col in required_resume_cols:
    if col not in resume_df.columns:
        raise ValueError(f'Missing resume column: {col}')
for col in required_job_cols:
    if col not in job_df.columns:
        raise ValueError(f'Missing job column: {col}')

resume_df = resume_df.drop_duplicates(subset=['resume_id']).reset_index(drop=True)
job_df = job_df.drop_duplicates(subset=['job_id']).reset_index(drop=True)

print('resume_df shape:', resume_df.shape)
print('job_df shape:', job_df.shape)
resume_df.head(2)


resume_df shape: (2481, 5)
job_df shape: (180370, 15)


,resume_id,category,resume,clean_resume,resume_word_count
0,16852973,HR,HR ADMINISTRATOR/MARKETING ASSOCIATE\...,hr administrator marketing associate hr admini...,658
1,22323967,HR,"HR SPECIALIST, US HR OPERATIONS ...",hr specialist us hr operations summary versati...,707


In [3]:

job_df.head(2)


,job_id,title,job,responsibilities,qualifications,skills,experience,clean_title,clean_job,clean_responsibilities,clean_qualifications,clean_skills,clean_experience,full_job,job_word_count
0,0,Digital Marketing Specialist,Social Media Managers oversee an organizations...,"Manage and grow social media accounts, create ...",M.Tech,"Social media platforms (e.g., Facebook, Twitte...",5 to 15 Years,digital marketing specialist,social media managers oversee an organizations...,manage and grow social media accounts create e...,m tech,social media platforms e g facebook twitter in...,5 to 15 years,digital marketing specialist digital marketing...,91
1,1,Web Developer,Frontend Web Developers design and implement u...,"Design and code user interfaces for websites, ...",BCA,"HTML, CSS, JavaScript Frontend frameworks (e.g...",2 to 12 Years,web developer,frontend web developers design and implement u...,design and code user interfaces for websites e...,bca,html css javascript frontend frameworks e g re...,2 to 12 years,web developer web developer frontend web devel...,83


In [4]:

resume_texts = resume_df['clean_resume'].fillna('').astype(str).tolist()
job_texts = job_df['full_job'].fillna('').astype(str).tolist()

# ---------- Dense semantic similarity ----------
resume_emb_path = MODEL_DIR / 'resume_embeddings.npy'
job_emb_path = MODEL_DIR / 'job_embeddings.npy'

if resume_emb_path.exists() and job_emb_path.exists():
    resume_dense = np.load(resume_emb_path)
    job_dense = np.load(job_emb_path)
    similarity_source = 'precomputed_sentence_transformer_embeddings'
    print('Loaded precomputed embeddings:', resume_dense.shape, job_dense.shape)
else:
    print('Precomputed embeddings not found. Using TF-IDF + TruncatedSVD dense semantic fallback.')
    semantic_vectorizer = TfidfVectorizer(
        max_features=20000,
        ngram_range=(1, 2),
        stop_words='english',
        min_df=2,
        max_df=0.95,
        sublinear_tf=True,
        norm='l2',
        dtype=np.float32,
    )
    combined_texts = resume_texts + job_texts
    combined_tfidf = semantic_vectorizer.fit_transform(combined_texts)
    n_components = min(256, max(16, combined_tfidf.shape[1] - 1))
    svd = TruncatedSVD(n_components=n_components, random_state=42)
    combined_dense = svd.fit_transform(combined_tfidf)
    combined_dense = normalize(combined_dense)
    resume_dense = combined_dense[:len(resume_df)].astype(np.float32)
    job_dense = combined_dense[len(resume_df):].astype(np.float32)
    similarity_source = 'svd_dense_fallback'
    print('Dense fallback shapes:', resume_dense.shape, job_dense.shape)

# ---------- Sparse TF-IDF similarity ----------
resume_tfidf_path = MODEL_DIR / 'tfidf_resume_vectors.npz'
job_tfidf_path = MODEL_DIR / 'tfidf_job_vectors.npz'
vectorizer_path = MODEL_DIR / 'tfidf_vectorizer.pkl'

if resume_tfidf_path.exists() and job_tfidf_path.exists():
    resume_tfidf = load_npz(resume_tfidf_path)
    job_tfidf = load_npz(job_tfidf_path)
    tfidf_source = 'precomputed_tfidf_artifacts'
    print('Loaded precomputed TF-IDF matrices:', resume_tfidf.shape, job_tfidf.shape)
else:
    print('Precomputed TF-IDF artifacts not found. Building TF-IDF on the fly.')
    tfidf_vectorizer = TfidfVectorizer(
        max_features=20000,
        ngram_range=(1, 2),
        stop_words='english',
        min_df=2,
        max_df=0.95,
        sublinear_tf=True,
        norm='l2',
        dtype=np.float32,
    )
    combined_texts = resume_texts + job_texts
    combined_sparse = tfidf_vectorizer.fit_transform(combined_texts)
    resume_tfidf = combined_sparse[:len(resume_df)]
    job_tfidf = combined_sparse[len(resume_df):]
    tfidf_source = 'tfidf_built_in_notebook'
    print('Built TF-IDF matrices:', resume_tfidf.shape, job_tfidf.shape)


Loaded precomputed embeddings: (2481, 384) (180370, 384)
Loaded precomputed TF-IDF matrices: (2481, 18916) (180370, 18916)


In [5]:

RANDOM_SEED = 42
TOP_K_HIGH = 3
TOP_K_MID = 2
TOP_K_LOW = 3

rng = np.random.default_rng(RANDOM_SEED)


def top_indices(scores: np.ndarray, k: int) -> np.ndarray:
    if scores.size == 0:
        return np.array([], dtype=int)
    k = min(k, scores.size)
    idx = np.argpartition(scores, -k)[-k:]
    idx = idx[np.argsort(scores[idx])[::-1]]
    return idx

pairs = []
num_jobs = len(job_df)
all_job_indices = np.arange(num_jobs)

for ridx in range(len(resume_df)):
    dense_scores = job_dense @ resume_dense[ridx]
    sparse_scores = (resume_tfidf[ridx] @ job_tfidf.T).toarray().ravel()

    dense_scaled = np.clip(dense_scores, 0, 1)
    sparse_scaled = np.clip(sparse_scores, 0, 1)
    hybrid_retrieval = 0.65 * dense_scaled + 0.35 * sparse_scaled

    high_idx = top_indices(hybrid_retrieval, TOP_K_HIGH)

    sorted_idx = np.argsort(hybrid_retrieval)[::-1]
    mid_pool = sorted_idx[10: min(len(sorted_idx), 60)]
    if len(mid_pool) == 0:
        mid_pool = sorted_idx[TOP_K_HIGH: min(len(sorted_idx), TOP_K_HIGH + 50)]
    mid_idx = rng.choice(mid_pool, size=min(TOP_K_MID, len(mid_pool)), replace=False) if len(mid_pool) else np.array([], dtype=int)

    low_pool = sorted_idx[int(0.75 * len(sorted_idx)):]
    if len(low_pool) == 0:
        low_pool = np.setdiff1d(all_job_indices, high_idx, assume_unique=False)
    low_idx = rng.choice(low_pool, size=min(TOP_K_LOW, len(low_pool)), replace=False) if len(low_pool) else np.array([], dtype=int)

    candidate_indices = []
    for bucket_name, arr in [('high', high_idx), ('mid', mid_idx), ('low', low_idx)]:
        for jidx in np.asarray(arr, dtype=int).tolist():
            candidate_indices.append((bucket_name, int(jidx)))

    seen = set()
    for bucket_name, jidx in candidate_indices:
        if jidx in seen:
            continue
        seen.add(jidx)

        feature_dict = build_pair_features(
            resume_df.iloc[ridx],
            job_df.iloc[jidx],
            embedding_similarity=float(dense_scaled[jidx]),
            tfidf_similarity=float(sparse_scaled[jidx]),
        )
        silver_score = hybrid_label_score(feature_dict)
        label, label_name = assign_multiclass_label(silver_score, bucket_name)

        pairs.append({
            'resume_id': int(resume_df.iloc[ridx]['resume_id']),
            'job_id': int(job_df.iloc[jidx]['job_id']),
            'resume_category': resume_df.iloc[ridx]['category'],
            'job_title': job_df.iloc[jidx]['title'],
            'retrieval_bucket': bucket_name,
            'silver_score': round(float(silver_score), 6),
            'label': int(label),
            'label_name': label_name,
            'matched_required_skills': ', '.join(feature_dict.pop('matched_required_skills')),
            'missing_required_skills': ', '.join(feature_dict.pop('missing_required_skills')),
            **feature_dict,
        })

pairs_df = pd.DataFrame(pairs)
print('Raw pairs shape:', pairs_df.shape)
pairs_df['label_name'].value_counts()


Raw pairs shape: (19848, 32)


label_name
poor_match        16473
moderate_match     3263
strong_match        112
Name: count, dtype: int64

In [6]:
# Recalibrate labels using retrieval-bucket-aware score quantiles
pairs_df = pairs_df.copy()

high_mask = pairs_df['retrieval_bucket'].eq('high')
mid_mask  = pairs_df['retrieval_bucket'].eq('mid')
low_mask  = pairs_df['retrieval_bucket'].eq('low')

high_cut = pairs_df.loc[high_mask, 'silver_score'].quantile(0.60) if high_mask.any() else 0.60
mid_cut  = pairs_df.loc[mid_mask,  'silver_score'].quantile(0.45) if mid_mask.any()  else 0.40
low_cut  = pairs_df.loc[low_mask,  'silver_score'].quantile(0.92) if low_mask.any()  else 0.55

pairs_df['label'] = 0
pairs_df.loc[high_mask, 'label'] = np.where(pairs_df.loc[high_mask, 'silver_score'] >= high_cut, 2, 1)
pairs_df.loc[mid_mask,  'label'] = np.where(pairs_df.loc[mid_mask,  'silver_score'] >= mid_cut,  1, 0)
pairs_df.loc[low_mask,  'label'] = np.where(pairs_df.loc[low_mask,  'silver_score'] >= low_cut,  1, 0)

# Safety net: guarantee some strong samples even on conservative scores
if (pairs_df['label'] == 2).sum() == 0 and high_mask.any():
    fallback_n = max(25, int(high_mask.sum() * 0.20))
    strong_idx = pairs_df.loc[high_mask].sort_values('silver_score', ascending=False).head(fallback_n).index
    pairs_df.loc[strong_idx, 'label'] = 2
    pairs_df.loc[high_mask & ~pairs_df.index.isin(strong_idx), 'label'] = 1

pairs_df['label_name'] = pairs_df['label'].map({0: 'poor_match', 1: 'moderate_match', 2: 'strong_match'})

# Optional balancing for cleaner training
class_counts = pairs_df['label'].value_counts().sort_index()
min_count    = class_counts.min()
print('Before balancing:')
print(class_counts)

balanced_parts = []
for label_value, group in pairs_df.groupby('label'):
    if len(group) > min_count:
        balanced_parts.append(group.sample(n=min_count, random_state=42))
    else:
        balanced_parts.append(group.copy())
balanced_df = pd.concat(balanced_parts, axis=0).sample(frac=1.0, random_state=42).reset_index(drop=True)
balanced_df['label_name'] = balanced_df['label'].map({0: 'poor_match', 1: 'moderate_match', 2: 'strong_match'})

print()
print('After balancing:')
print(balanced_df['label_name'].value_counts())
balanced_df.head(3)


Before balancing:
label
0    9080
1    7791
2    2977
Name: count, dtype: int64

After balancing:
label_name
moderate_match    2977
poor_match        2977
strong_match      2977
Name: count, dtype: int64


,resume_id,job_id,resume_category,job_title,retrieval_bucket,silver_score,label,label_name,matched_required_skills,missing_required_skills,...,job_education_rank,education_match_score,required_skill_coverage,preferred_skill_coverage,missing_required_skill_ratio,missing_required_skill_count,skill_match_count,project_relevance_score,certification_relevance_score,job_skill_count
0,91189201,26494,CONSULTANT,Marketing Manager,high,0.243791,1,moderate_match,,product positioning and messaging market resea...,...,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0
1,39064638,154492,ARTS,Sales Manager,high,0.200985,1,moderate_match,,sales leadership team management sales strateg...,...,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0
2,90318913,22306,AVIATION,Electrical Engineer,mid,0.221725,0,poor_match,,electrical engineering power system analysis r...,...,3.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0


In [7]:
# ── Add 10 engineered features to balanced_df before saving ──────────────────
#
# These are derived purely from columns already in balanced_df.
# Adding them HERE means the CSV is the single source of truth —
# run_multiclass_models.ipynb just loads and trains, no extra computation.
#

# 1. experience_gap — signed difference, clipped to [-10, +10]
balanced_df['experience_gap'] = (
    balanced_df['resume_years_detected'] - balanced_df['job_min_years']
).clip(-10, 10)

# 2. experience_deficit_flag — 1 when candidate has fewer years than required
balanced_df['experience_deficit_flag'] = (
    balanced_df['experience_gap'] < 0
).astype(float)

# 3. skill_density — skill coverage normalised by job complexity
balanced_df['skill_density'] = (
    balanced_df['required_skill_coverage'] /
    balanced_df['job_skill_count'].replace(0, float('nan'))
).fillna(0).clip(0, 1)

# 4. semantic_skill_gap — high embedding sim BUT high missing skills
balanced_df['semantic_skill_gap'] = (
    balanced_df['embedding_similarity'] *
    balanced_df['missing_required_skill_ratio']
).clip(0, 1)

# 5. hybrid_match_score — calibrated weighted blend of best features
balanced_df['hybrid_match_score'] = (
    0.30 * balanced_df['required_skill_coverage'] +
    0.20 * balanced_df['experience_match_score']  +
    0.15 * balanced_df['education_match_score']   +
    0.20 * balanced_df['embedding_similarity']    +
    0.10 * balanced_df['keyword_overlap']         +
    0.05 * balanced_df['title_match_score']
).clip(0, 1)

# 6. exp_x_skills — interaction: counts only when BOTH experience AND skills match
balanced_df['exp_x_skills'] = (
    balanced_df['experience_match_score'] *
    balanced_df['required_skill_coverage']
).clip(0, 1)

# 7. missing_log — log-scaled missing skill count
balanced_df['missing_log'] = np.log1p(
    balanced_df['missing_required_skill_count']
)

# 8. emb_x_tfidf — product of both semantic signals
balanced_df['emb_x_tfidf'] = (
    balanced_df['embedding_similarity'] *
    balanced_df['tfidf_similarity']
).clip(0, 1)

# 9. emb_sq — squared embedding similarity
balanced_df['emb_sq'] = balanced_df['embedding_similarity'] ** 2

# 10. exp_sq — squared experience match score
balanced_df['exp_sq'] = balanced_df['experience_match_score'] ** 2

NEW_FEATURE_COLS = [
    'experience_gap', 'experience_deficit_flag', 'skill_density',
    'semantic_skill_gap', 'hybrid_match_score', 'exp_x_skills',
    'missing_log', 'emb_x_tfidf', 'emb_sq', 'exp_sq',
]

print(f'New features added: {len(NEW_FEATURE_COLS)}')
print(f'balanced_df shape now: {balanced_df.shape}')
print('Sample values (first row):')
print(balanced_df[NEW_FEATURE_COLS].iloc[0].round(4))


New features added: 10
balanced_df shape now: (8931, 42)
Sample values (first row):
experience_gap            -10.0000
experience_deficit_flag     1.0000
skill_density               0.0000
semantic_skill_gap          0.4611
hybrid_match_score          0.2989
exp_x_skills                0.0000
missing_log                 0.6931
emb_x_tfidf                 0.0672
emb_sq                      0.2127
exp_sq                      0.0000
Name: 0, dtype: float64


In [8]:

output_path = SCORE_DIR / 'evaluation_pairs_multiclass.csv'
balanced_df.to_csv(output_path, index=False)
print('Saved:', output_path)

# Gold template for manual validation
gold_parts = []
for label_name, group in balanced_df.groupby('label_name'):
    sample_n = min(120, len(group))
    gold_parts.append(group.sample(n=sample_n, random_state=42))
gold_template = pd.concat(gold_parts, axis=0).reset_index(drop=True)

gold_template.insert(len(gold_template.columns), 'manual_label', '')
gold_template.insert(len(gold_template.columns), 'review_notes', '')
gold_template_path = SCORE_DIR / 'evaluation_pairs_gold_template.csv'
gold_template.to_csv(gold_template_path, index=False)
print('Saved:', gold_template_path)

summary = {
    'num_resumes': int(len(resume_df)),
    'num_jobs': int(len(job_df)),
    'raw_pairs': int(len(pairs_df)),
    'balanced_pairs': int(len(balanced_df)),
    'label_distribution': balanced_df['label_name'].value_counts().to_dict(),
    'feature_columns': list(FEATURE_COLUMNS) + NEW_FEATURE_COLS,
    'semantic_similarity_source': similarity_source,
    'tfidf_source': tfidf_source,
}
summary_path = SCORE_DIR / 'pair_generation_summary.json'
with open(summary_path, 'w', encoding='utf-8') as f:
    json.dump(summary, f, indent=2)
print('Saved:', summary_path)
summary


Saved: C:\Users\Mr_Root\Downloads\New folder (4)\myproject\data\score\evaluation_pairs_multiclass.csv
Saved: C:\Users\Mr_Root\Downloads\New folder (4)\myproject\data\score\evaluation_pairs_gold_template.csv
Saved: C:\Users\Mr_Root\Downloads\New folder (4)\myproject\data\score\pair_generation_summary.json


{'num_resumes': 2481,
 'num_jobs': 180370,
 'raw_pairs': 19848,
 'balanced_pairs': 8931,
 'label_distribution': {'moderate_match': 2977,
  'poor_match': 2977,
  'strong_match': 2977},
 'feature_columns': ['embedding_similarity',
  'tfidf_similarity',
  'semantic_hint',
  'keyword_overlap',
  'title_match_score',
  'resume_word_count',
  'job_word_count',
  'length_ratio',
  'resume_years_detected',
  'job_min_years',
  'experience_match_score',
  'resume_education_rank',
  'job_education_rank',
  'education_match_score',
  'required_skill_coverage',
  'preferred_skill_coverage',
  'missing_required_skill_ratio',
  'missing_required_skill_count',
  'skill_match_count',
  'project_relevance_score',
  'certification_relevance_score',
  'job_skill_count',
  'experience_gap',
  'experience_deficit_flag',
  'skill_density',
  'semantic_skill_gap',
  'hybrid_match_score',
  'exp_x_skills',
  'missing_log',
  'emb_x_tfidf',
  'emb_sq',
  'exp_sq'],
 'semantic_similarity_source': 'precomputed_s